# Textual Embeddings Adventure Starts Here

Our first goal should be to understand whether generic sentence embeddings can recover sensible connections between the candidate-test statements/questions and parliamentary business.

Conceptually, what we are doing is a semantic retrieval problem:

Candidate-test question → search through parliamentary votes → retrieve the most semantically related votes.

https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html

In [53]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/temp_data")

roll_calls_file = DATA_DIR / "parliament" / "roll_calls_resume.csv"
candidate_test_file = DATA_DIR / "candidate_test" / "Kandidattestdata.xlsx"

df_roll_calls = pd.read_csv(roll_calls_file)
df_FV11 = pd.read_excel(candidate_test_file, sheet_name="FV11")
df_FV15 = pd.read_excel(candidate_test_file, sheet_name="FV15")
df_FV19 = pd.read_excel(candidate_test_file, sheet_name="FV19")
df_FV22 = pd.read_excel(candidate_test_file, sheet_name="FV22")

print(df_roll_calls.shape)
print(df_roll_calls.columns.tolist())
print(df_FV11.shape)
print(df_FV11.columns.tolist())

(2128, 12)
['afstemningid', 'sagstrinid', 'kommentar', 'afstemningstypeid', 'dato', 'sagstrintypeid', 'sagid', 'sagstypeid', 'sag_nummer', 'sag_titel', 'sag_titelkort', 'sag_resume']
(14540, 12)
['id', 'Firstname', 'Lastname', 'Candidates.Party', 'Storkreds', 'gender', 'birthdate', 'Question', 'Answer', 'Answer (Text)', 'IsImportant', 'Comment']


In [54]:
df_FV22.columns.tolist()

['id',
 'Candidates.Firstname',
 'Candidates.Lastname',
 'Candidates.Party',
 'Candidates.Area',
 'gender',
 'birthdate',
 'Question',
 'Answer',
 'Answer (text)',
 'IsImportant',
 'Comment']

In [55]:
# We don't need duplicate questions, so we can drop them and reset the index
df_questions_FV22 = (
    df_FV22[["Question"]]
    .drop_duplicates()
    .dropna()
    .reset_index(drop=True)
)

print(df_questions_FV22.shape)

df_questions_FV22.head(20)

(51, 1)


,Question
0,Afgiften på såkaldte hybridbiler bør være den ...
1,Aldersgrænsen for at købe øl og vin skal hæves...
2,Asylansøgere bør sendes til et land uden for E...
3,Behandling hos tandlægen bør være gratis for b...
4,Brugen af ukrudtsmidlet Roundup bør forbydes i...
5,Danmark bør indføre CO2-afgift på flyrejser
6,Danmark bør tage imod flere kvoteflygtninge
7,Danmark skal bruge flere penge på at styrke to...
8,Danmark skal undersøge muligheden for at udvik...
9,Den såkaldte Arne-pension skal afskaffes


In [56]:
df_questions_FV22["election"] = "FV22"

In [57]:
def get_unique_questions(df, election):
    out = (
        df[["Question"]]
        .drop_duplicates()
        .dropna()
        .reset_index(drop=True)
    )
    
    out["election"] = election
    return out


df_questions_FV11 = get_unique_questions(df_FV11, "FV11")
df_questions_FV15 = get_unique_questions(df_FV15, "FV15")
df_questions_FV19 = get_unique_questions(df_FV19, "FV19")
df_questions_FV22 = get_unique_questions(df_FV22, "FV22")

In [58]:
# Unique questions per election
print("FV11:", len(df_questions_FV11))
print("FV15:", len(df_questions_FV15))
print("FV19:", len(df_questions_FV19))
print("FV22:", len(df_questions_FV22))

FV11: 20
FV15: 20
FV19: 45
FV22: 51


In [59]:
df_questions = pd.concat(
    [
        df_questions_FV11,
        df_questions_FV15,
        df_questions_FV19,
        df_questions_FV22,
    ],
    ignore_index=True,
)

print(df_questions.shape)
df_questions.head()

(136, 2)


,Question,election
0,Den danske krone skal erstattes med euroen,FV11
1,EU skal fortsætte forhandlingerne med Tyrkiet ...,FV11
2,Kirke og stat skal adskilles,FV11
3,Christiansborg favoriserer hovedstadsområdet p...,FV11
4,"Folketinget skal fastsætte kvoter, så der alti...",FV11


In [60]:
df_questions["Question"].duplicated().sum()

np.int64(6)

In [61]:
df_questions[
    df_questions["Question"].duplicated(keep=False)
].sort_values("Question")

,Question,election
22,Afgiften på cigaretter skal sættes op,FV15
40,Afgiften på cigaretter skal sættes op,FV19
33,Den offentlige kulturstøtte skal sænkes,FV15
75,Den offentlige kulturstøtte skal sænkes,FV19
2,Kirke og stat skal adskilles,FV11
27,Kirke og stat skal adskilles,FV15
82,Kirke og stat skal adskilles,FV19
69,Kriminalitet begået i udsatte boligområder ska...,FV19
112,Kriminalitet begået i udsatte boligområder ska...,FV22
7,Offentlige institutioner i Danmark tager for m...,FV11


In [62]:
df_cases = (
    df_roll_calls[
        [
            "sagid",
            "sag_nummer",
            "sag_titel",
            "sag_titelkort",
            "sag_resume",
            "dato",
        ]
    ]
    .drop_duplicates(subset="sagid")
    .reset_index(drop=True)
)

In [63]:
df_cases["text"] = (
    df_cases["sag_titel"].fillna("")
    + " "
    + df_cases["sag_resume"].fillna("")
)

In [64]:
pd.set_option("display.max_colwidth", None)

df_cases[
    [
        "sag_nummer",
        "sag_titel",
        "sag_resume",
        "text"
    ]
].head(10)

,sag_nummer,sag_titel,sag_resume,text
0,L 200,Forslag til lov om ændring af virksomhedsskatteloven og kildeskatteloven. (Indgreb mod utilsigtet udnyttelse af virksomhedsordningen ved indskud af privat gæld m.v.).,"Loven ændrer virksomhedsskatteordningens regler, så det sikres, at selvstændigt erhvervsdrivende ikke kan udnytte virksomhedsskatteordningen utilsigtet. Med loven sikres det, at selvstændige ikke kan anvende lavt beskattede midler til at finansiere privatforbrug og afdrage på privat gæld, uden at midlerne beskattes som personlig indkomst. Der ændres ikke på virksomhedsskatteordningens grundlæggende struktur og indhold.\n\nLoven indeholder følgende elementer:\n- Selvstændige kan fremover kun spare op i virksomhedsordningen, hvis indskudskontoen er nul eller positiv. \n- Hvis aktiver, der indgår i virksomhedsordningen, fremover stilles til sikkerhed for gæld, der ikke indgår i virksomhedsordningen, anses et tilsvarende beløb for hævet af den selvstændige.\n- Rentekorrektionen forhøjes effektivt med 3 procentpoint med henblik på at eliminere den skattemæssige besparelse, som selvstændige kan opnå ved at placere private renteudgifter i virksomhedsordningen.\n- For selvstændige, der ved lovforslagets fremsættelse anvender virksomhedsordningen, og som allerede har stillet virksomhedens aktiver til sikkerhed for gæld, der ikke indgår i ordningen, eller har en negativ indskudskonto, suspenderes muligheden for at spare op i ordningen. Det gælder dog kun, hvis summen af den nominelle værdi af den negative indskudskonto og en evt. sikkerhedsstillelse overstiger 100.000 kr.\n\nBegrænsningerne i forhold til opsparingsmuligheden og forhøjelsen af rentekorrektionssatsen skønnes at indebære et umiddelbart merprovenu på ca. 0,8 mia. kr. i 2014, ca. 0,9 mia.kr. i 2015 og ca. 0,8 mia. kr. fra og med 2016. Den varige umiddelbare provenuvirkning skønnes at udgøre ca. 0,65 mia. kr. og ca. 0,5 mia. kr. efter tilbageløb.\n\nFor hurtigst muligt at give selvstændige klarhed om de nye regler, og da dele af loven skulle have virkning allerede fra og med fremsættelsen af lovforslaget af hensyn til indgrebets effektivitet og hindring af hamstring, er loven blevet vedtaget i indeværende folketingsår.","Forslag til lov om ændring af virksomhedsskatteloven og kildeskatteloven. (Indgreb mod utilsigtet udnyttelse af virksomhedsordningen ved indskud af privat gæld m.v.). Loven ændrer virksomhedsskatteordningens regler, så det sikres, at selvstændigt erhvervsdrivende ikke kan udnytte virksomhedsskatteordningen utilsigtet. Med loven sikres det, at selvstændige ikke kan anvende lavt beskattede midler til at finansiere privatforbrug og afdrage på privat gæld, uden at midlerne beskattes som personlig indkomst. Der ændres ikke på virksomhedsskatteordningens grundlæggende struktur og indhold.\n\nLoven indeholder følgende elementer:\n- Selvstændige kan fremover kun spare op i virksomhedsordningen, hvis indskudskontoen er nul eller positiv. \n- Hvis aktiver, der indgår i virksomhedsordningen, fremover stilles til sikkerhed for gæld, der ikke indgår i virksomhedsordningen, anses et tilsvarende beløb for hævet af den selvstændige.\n- Rentekorrektionen forhøjes effektivt med 3 procentpoint med henblik på at eliminere den skattemæssige besparelse, som selvstændige kan opnå ved at placere private renteudgifter i virksomhedsordningen.\n- For selvstændige, der ved lovforslagets fremsættelse anvender virksomhedsordningen, og som allerede har stillet virksomhedens aktiver til sikkerhed for gæld, der ikke indgår i ordningen, eller har en negativ indskudskonto, suspenderes muligheden for at spare op i ordningen. Det gælder dog kun, hvis summen af den nominelle værdi af den negative indskudskonto og en evt. sikkerhedsstillelse overstiger 100.000 kr.\n\nBegrænsningerne i forhold til opsparingsmuligheden og forhøjelsen af rentekorrektionssatsen skønnes at indebære et umiddelbart merprovenu på ca. 0,8 mia. kr. i 2014, ca. 0,9 mia.kr. i 2015 og ca. 0,8 mia. kr. fra og med 2016

# We now have to df's: Questions and Cases. 

I will try a simple embedding model and try a single question against all 2,128 cases.

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "intfloat/multilingual-e5-base"
)

ModuleNotFoundError: No module named 'sentence_transformers'